# ReAct Agent

In [9]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import Runnable
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION
)


# Tools

In [7]:
# TOOL - 1 [News Search Tool]

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(description="This is a tool to search the web using DuckDuckGo. It can be used to find information on the internet that may not be available in the model's training data.")

# Tool - 2 [Wikipedia Search Tool]

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(), description="This is a tool to search Wikipedia. It can be used to find information on a wide range of topics that may not be available in the model's training data.")


# Tool - 3 [Custom Enterprise Tool]
from langchain.tools import tool

@tool
def enterprise_tool(query:str) -> str:

    """This is a tool to send emails to employees"""

    return "Email Sent"

ToolKit = [search_tool, wikipedia_tool, enterprise_tool]
ToolKit

[DuckDuckGoSearchRun(description="This is a tool to search the web using DuckDuckGo. It can be used to find information on the internet that may not be available in the model's training data.", api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')),
 WikipediaQueryRun(description="This is a tool to search Wikipedia. It can be used to find information on a wide range of topics that may not be available in the model's training data.", api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\10706030\\Desktop\\iamtest2\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='enterprise_tool', description='This is a tool to send emails to employees', args_schema=<class 'langchain_core.utils.pydantic.enterprise_tool'>, func=<function enterprise_tool at 0x000002124F53AFC0>)]

# ReAct Agent

In [17]:
from langchain.agents import create_agent

agent = create_agent(llm_openai, tools=ToolKit)
print(agent)

<langgraph.graph.state.CompiledStateGraph object at 0x0000021258009410>

# ReAct Agent Invoke With Streams

In [18]:
example_query = "Who is the president of Taiwan?"

events = agent.stream(
    {"messages":[("user",example_query)]},
    stream_mode="values"
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who is the president of Taiwan?


[04/08/26 17:18:03] INFO     HTTP Request: POST                                                     ]8;id=925283;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=22001;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:18:03] INFO     HTTP Request: POST                                                     ]8;id=326873;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=110236;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:18:03] INFO     HTTP Request: POST                                                     ]8;id=763376;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=854459;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

================================== Ai Message ==================================
Tool Calls:
  wikipedia (call_ICifB4ZtmmeM4KbO6q0mh2Au)
 Call ID: call_ICifB4ZtmmeM4KbO6q0mh2Au
  Args:
    query: President of Taiwan
================================= Tool Message =================================
Name: wikipedia

Page: President of the Republic of China
Summary: The president of the Republic of China, also known as the president of Taiwan, is the head of state of the Republic of China (Taiwan), as well as the commander-in-chief of the Republic of China Armed Forces. Before 1949 the position had the authority of ruling over mainland China, but losing control of it after communist victory in the Chinese Civil War, the remaining jurisdictions of the ROC have been limited to Taiwan, Penghu, Kinmen, Matsu, and smaller islands.
Originally elected by the National Assembly, the presidency was intended to be a ceremonial office with no real executive power because the ROC was originally envision

[04/08/26 17:18:18] INFO     HTTP Request: POST                                                     ]8;id=676943;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=483702;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:18:18] INFO     HTTP Request: POST                                                     ]8;id=388108;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=173347;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:18:18] INFO     HTTP Request: POST                                                     ]8;id=898643;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=489378;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

================================== Ai Message ==================================

The current president of Taiwan (Republic of China) is Lai Ching-te of the Democratic Progressive Party. He succeeded Tsai Ing-wen, who was also from the same party.


# Manually Binding The LLM With Tools

In [19]:
# Without Binding
llm_openai.invoke("What's the latest news about the stock market?")

[04/08/26 17:20:45] INFO     HTTP Request: POST                                                     ]8;id=39379;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=56251;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:20:45] INFO     HTTP Request: POST                                                     ]8;id=300544;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=79950;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:20:45] INFO     HTTP Request: POST                                                     ]8;id=500360;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=795390;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(content='I don’t have real-time access to financial news, but as of my last update in June 2024, major global stock markets have been volatile due to ongoing concerns about inflation, central bank policy decisions, and geopolitical tensions. The U.S. Federal Reserve has been closely watched for signals about interest rate adjustments, while tech stocks have generally performed well, buoyed by enthusiasm for artificial intelligence.\n\nFor the most current updates, I recommend checking trusted sources like Bloomberg, Reuters, CNBC, or your preferred financial news outlet. If you have questions about specific stocks or sectors, let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 16, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 

In [26]:
# With Binding
llm_binded = llm_openai.bind_tools(ToolKit)
response = llm_binded.invoke("What's the latest news about the stock market?")
print(response)


[04/08/26 17:30:24] INFO     HTTP Request: POST                                                     ]8;id=218265;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=564548;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:30:24] INFO     HTTP Request: POST                                                     ]8;id=841493;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=792031;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 17:30:24] INFO     HTTP Request: POST                                                     ]8;id=443466;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=459355;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 23,
            'prompt_tokens': 168,
            'total_tokens': 191,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4.1-2025-04-14',
        'system_fingerprint': 'fp_dc49d4fcba',
        'id': 'chatcmpl-DSJWnYwvlEelF2Z26h65H0qQUbC81',
        'prompt_filter_results': [
            {
                'prompt_index': 0,
                'content_filter_results': {
                    'hate': {'filtered': False, 'severity': 'safe'},
                    'jailbreak': {'detected': False, 'filtered': False},
                    'self_harm': {'filtered': False, 'severity': 'safe'},
                    'sexual': {'filtered': False, 'severity': 'safe'},
                    'violence': {'filtered': False, 'severity': 'safe'}
                }
            }
        ],
        'finish_reason': 'tool_calls',
        'logprobs': None,
        'content_filter_results': {}
    },
    id='lc_run--019d6c6d-ae6a-7d32-a1a7-9f0c9b43c012-0',
    tool_calls=[
        {
            'name': 'duckduckgo_search',
            'args': {'query': 'latest news about the stock market'},
            'id': 'call_swjKPNXzGVtfSj5IL5NHLyNC',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 168,
        'output_tokens': 23,
        'total_tokens': 191,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)